In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## Import Required Libraries

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

print("✅ Environment loaded")
print(f"Current directory:{os.getcwd()}")
print(f"Sample data directory exists:{Path('/content/drive/MyDrive/Datasets/sample_data').exists()}")

✅ Environment loaded
Current directory:/content
Sample data directory exists:True


In [3]:
from langchain_core.documents import Document

# Create a sample document
doc = Document(
    page_content='This is the actual content of the document. It contains the text we want to process.',
    metadata={
        "source":"example.pdf",
        "page":1,
        "author":"John Doe",
        "date":"2025-01-15"
    }
)

print("📄 Document Structure:")
print(f"\nType:{type(doc)}")
print(f"\nContent(first 100 chars):{doc.page_content[:100]}...")
print(f"\nMetadata:{doc.metadata}")
print(f"\nSource:{doc.metadata['source']}")
print(f"\nPage Number:{doc.metadata['page']}")

📄 Document Structure:

Type:<class 'langchain_core.documents.base.Document'>

Content(first 100 chars):This is the actual content of the document. It contains the text we want to process....

Metadata:{'source': 'example.pdf', 'page': 1, 'author': 'John Doe', 'date': '2025-01-15'}

Source:example.pdf

Page Number:1


# Loading PDF Files

In [4]:
#!pip install langchain_community

In [5]:
#!pip install pypdf

In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "/content/drive/MyDrive/Datasets/pdfs/rag.pdf"

if Path(pdf_path).exists():
  print(f"Loading PDF:{pdf_path}")
  print(f"⏳ This may take a moment ...\n")

  loader = PyPDFLoader(pdf_path)
  documents = loader.load()

  print(f"✅ Loaded {len(documents)} pages\n")
  print(f"📄 First Page:")
  print(f"   Content(first 200 chars):{documents[0].page_content[:200]}...")
  print(f"\n Metadata:{documents[0].metadata}")

  print(f"\n📄 Last Page(page {len(documents)}):")
  print(f"     Content (first 200 chars):{documents[-1].page_content[:200]}...")
else:
  print(f"❌ PDF not found: {pdf_path}")
  print("   Make sure the file exists in the project root")

Loading PDF:/content/drive/MyDrive/Datasets/pdfs/rag.pdf
⏳ This may take a moment ...

✅ Loaded 19 pages

📄 First Page:
   Content(first 200 chars):Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, W...

 Metadata:{'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/drive/MyDrive/Datasets/pdfs/rag.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}

📄 Last Page(page 19):
     Content (first 200 chars):Table 7: Number of instances in the datasets used. *A hidden subset of this data is used for evaluation
Task Train Development Test
Natural

# Lazy Loading for large PDF's

In [7]:
if Path(pdf_path).exists():
  loader = PyPDFLoader(pdf_path)

  print("🔄 Lazy loading pages (memory efficient):")

  for i, page in enumerate(loader.lazy_load()):
    if i>=5: # Load only first 3 pages
      break

    print(f"\nPage:{i+1}:")
    print(f"     Length:{len(page.page_content)} charecters")
    print(f"     Preview:{page.page_content[:100]}...")

  print("\n💡 Tip: Use lazy_load() for PDFs >100 pages to save memory")

🔄 Lazy loading pages (memory efficient):

Page:1:
     Length:2899 charecters
     Preview:Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Alek...

Page:2:
     Length:4545 charecters
     Preview:The	Divine
Comedy	(x) q
Query
Encoder
q(x)
MIPS pθ
Generator pθ
(Parametric)
Margin-
alize
This	14th...

Page:3:
     Length:3655 charecters
     Preview:byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original...

Page:4:
     Length:4205 charecters
     Preview:minimize the negative marginal log-likelihood of each target,∑
j− logp(yj|xj) using stochastic
gradi...

Page:5:
     Length:4556 charecters
     Preview:MSMARCO as an open-domain abstractive QA task. MSMARCO has some questions that cannot be
answered in...

💡 Tip: Use lazy_load() for PDFs >100 pages to save memory


##Loading multiple pages from a directory

In [8]:
pdf_directory = "/content/drive/MyDrive/Datasets/pdfs"

if Path(pdf_directory).exists():
  print(f"📂 Loading PDFs from:{pdf_directory}/\n")

  all_documents=[]

  # Find all pdf files
  pdf_files=list(Path(pdf_directory).glob("*.pdf"))
  print(f"Found {len(pdf_files)} PDF files:")

  for pdf_file in pdf_files:
    print(f"    -{pdf_file.name}")

    # Load each PDF
    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()
    all_documents.extend(docs)

    print(f"    ✅ Loaded {len(docs)} pages")

  print(f"\n📊 Total:{len(all_documents)} pages from {len(pdf_files)} PDFs")

  # Show unique sources
  sources = set(doc.metadata['source'] for doc in all_documents)
  print(f"\n Sources:")
  for source in sources:
    print(f"    -{Path(source).name}")

else:
  print(f"❌ Directory not found:{pdf_directory}")

📂 Loading PDFs from:/content/drive/MyDrive/Datasets/pdfs/

Found 2 PDF files:
    -ragsurvey.pdf
    ✅ Loaded 21 pages
    -rag.pdf
    ✅ Loaded 19 pages

📊 Total:40 pages from 2 PDFs

 Sources:
    -rag.pdf
    -ragsurvey.pdf


# Loading CSV Files

In [9]:
from langchain_community.document_loaders import CSVLoader

csv_path = "/content/drive/MyDrive/Datasets/sample_data/products.csv"

if Path(csv_path).exists():
  print(f"Loading CSV:{csv_path}\n")

  loader = CSVLoader(
      file_path=csv_path,
      source_column="product_name",
  )

  # Load all rows
  documents = loader.load()
  print(f"✅ Loaded {len(documents)} products\n")

  # Inspect first 3 products
  for i, doc in enumerate(documents[:3], 1):
    print(f"{"="*70}")
    print(f"Product {i}:")
    print(f"{"="*70}")
    print(doc.page_content)
    print(f"\nSource:{doc.metadata['source']}")
    print(f"Row:{doc.metadata.get('row', 'N/A')}")
    print()

  print(f"... and {len(documents)-3} more products")
else:
  print(f"❌ CSV not found:{csv_path}")

Loading CSV:/content/drive/MyDrive/Datasets/sample_data/products.csv

✅ Loaded 15 products

Product 1:
product_id: 1
product_name: Laptop Pro 15
category: Electronics
description: High-performance laptop with 15-inch display, Intel i7 processor, 16GB RAM, and 512GB SSD. Perfect for professional work and gaming.
price: 1299.99
stock: 45

Source:Laptop Pro 15
Row:0

Product 2:
product_id: 2
product_name: Wireless Mouse
category: Accessories
description: Ergonomic wireless mouse with 6 programmable buttons, 2400 DPI optical sensor, and long battery life. Compatible with Windows and Mac.
price: 29.99
stock: 150

Source:Wireless Mouse
Row:1

Product 3:
product_id: 3
product_name: USB-C Hub
category: Accessories
description: 7-in-1 USB-C hub with HDMI, USB 3.0 ports, SD card reader, and USB-C power delivery. Ideal for laptops and tablets.
price: 49.99
stock: 80

Source:USB-C Hub
Row:2

... and 12 more products


In [10]:
if Path(csv_path).exists():
  # Advanced CSV Loading with custom configuration
  loader = CSVLoader(
      file_path = csv_path,
      csv_args = {
          'delimiter':',',
          'quotechar':'"',
          'fieldnames':None
      },
      source_column = "product_id"

  )
  docs = loader.load()
  # Show how metadata is different
  print("📊 CSV with custom configuration:\n")
  print(f"First document source:{docs[0].metadata['source']}")
  print(f"Content preview:\n{docs[0].page_content[:200]}...")

📊 CSV with custom configuration:

First document source:1
Content preview:
product_id: 1
product_name: Laptop Pro 15
category: Electronics
description: High-performance laptop with 15-inch display, Intel i7 processor, 16GB RAM, and 512GB SSD. Perfect for professional work an...


# Loading JSON Files

In [11]:
pip install jq

In [12]:
from langchain_community.document_loaders import JSONLoader

json_path = '/content/drive/MyDrive/Datasets/sample_data/api_response.json'

if Path(json_path).exists():
  print(f"Loading JSON:{json_path}\n")

  loader = JSONLoader(
      file_path = json_path,
      jq_schema = ".articles[]",
      text_content = False
  )

  # Load articles

  documents = loader.load()

  print(f"✅ Loaded {len(documents)} articles\n")

  # Inspect first article

  print("📰 First Article:")
  print(f"Content:\n{documents[0].page_content}\n")
  print(f"Metadata:{documents[0].metadata}")

else:
  print(f"❌ JSON not found:{json_path}")

Loading JSON:/content/drive/MyDrive/Datasets/sample_data/api_response.json

✅ Loaded 5 articles

📰 First Article:
Content:
{"id": "article_001", "title": "Introduction to Retrieval-Augmented Generation (RAG)", "author": "Dr. Sarah Chen", "published_date": "2025-01-10", "category": "Machine Learning", "tags": ["RAG", "LLM", "NLP", "AI"], "summary": "Retrieval-Augmented Generation (RAG) is a powerful technique that combines information retrieval with large language models to generate more accurate and contextual responses.", "content": "RAG systems work by first retrieving relevant documents from a knowledge base, then using those documents as context for a language model to generate responses. This approach significantly reduces hallucinations and provides more factual, grounded outputs. The architecture typically consists of three main components: a document store, an embedding model for semantic search, and a language model for generation.", "reading_time": "5 minutes", "views": 15420

## Extracting specific fields from JSON

In [13]:
if Path(json_path).exists():

  loader = JSONLoader(
      file_path = json_path,
      jq_schema = ".articles[].content",
      text_content = True
  )

  docs = loader.load()

  print(f"📝 Extracted article contents only:\n")

  for i, doc in enumerate(docs[:2], 1):
    print(f"{i}.{doc.page_content[:150]}...")
    print()

📝 Extracted article contents only:

1.RAG systems work by first retrieving relevant documents from a knowledge base, then using those documents as context for a language model to generate ...

2.Vector databases like FAISS, Pinecone, and Chroma provide optimized storage and retrieval for embedding vectors. Unlike traditional databases that use...



# Loading web pages (HTML)

## Loading a local HTML File

In [14]:
!pip install unstructured

In [15]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

html_path = '/content/drive/MyDrive/Datasets/sample_data/blog_post.html'

if Path(html_path).exists():
  print(f"Loading HTML:{html_path}\n")
  # For local files, we need to use file:// protocol
  file_url = f"file://{Path(html_path).absolute()}"

  # Create Loader
  loader = UnstructuredHTMLLoader(html_path)

  # Load the page
  documents = loader.load()
  print(f"✅ Loaded {len(documents)} document(s)\n")

  # Inspect Content
  doc = documents[0]
  print(f"📄 Content length:{len(doc.page_content)} characters")
  print(f"\n📝 First 500 characters:\n{doc.page_content[:500]}...")
  print(f"\n🔍 Metadata:{doc.metadata}")
else:
  print(f"❌ HTML not found:{html_path}")

Loading HTML:/content/drive/MyDrive/Datasets/sample_data/blog_post.html

✅ Loaded 1 document(s)

📄 Content length:7197 characters

📝 First 500 characters:
Building Intelligent Applications with RAG

By Dr. Amanda Foster | January 15, 2025 | 12 min read

Introduction

In the rapidly evolving landscape of artificial intelligence, Retrieval-Augmented Generation (RAG) has emerged as a game-changing approach for building intelligent applications. Unlike traditional chatbots that rely solely on the knowledge embedded in their training data, RAG systems combine the power of information retrieval with language generation to produce more accurate, contextu...

🔍 Metadata:{'source': '/content/drive/MyDrive/Datasets/sample_data/blog_post.html'}


## Loading multiple URLs

In [16]:
from langchain_community.document_loaders import WebBaseLoader

urls =[
        "https://python.langchain.com/docs/introduction/",
        "https://python.langchain.com/docs/expression_language/"
]

loader = WebBaseLoader(urls)
docs = loader.load()
print(f"Loaded {len(docs)} pages")
for doc in docs:
  print(f"    -{doc.metadata['source']}")
print(f"💡 WebBaseLoader Example:")
print(f"\nTo load web pages, use:")
print("\n⚠️ Note:Only works with static HTML(no JavaScript rendering)")

Loaded 2 pages
    -https://python.langchain.com/docs/introduction/
    -https://python.langchain.com/docs/expression_language/
💡 WebBaseLoader Example:

To load web pages, use:

⚠️ Note:Only works with static HTML(no JavaScript rendering)


In [17]:
# Print content from both loaded pages
print("="*80)
print("📄 LOADED DOCUMENTS CONTENT")
print("="*80)

for i, doc in enumerate(docs, 1):
  print(f"\n{"="*80}")
  print(f"📄 PAGE {i}:{doc.metadata['source']}")
  print(f"{'='*80}")

  # Print first 1000 characters of the content
  print(f"\n📝 Content preview (first 1000 chars):")
  print(doc.page_content[:1000])
  print(f"\n...[Total length:{len(doc.page_content)} characters]")

  # Print metadata
  print(f"\n🔍 Metadata:")
  for key, value in doc.metadata.items():
    print(f"    {key}:{value}")
  print("\n")

# Full content of a specific page
print("\n"+"="*80)
print("📖 FULL CONTENT OF PAGE 1")
print("="*80)
print(docs[0].page_content)

📄 LOADED DOCUMENTS CONTENT

📄 PAGE 1:https://python.langchain.com/docs/introduction/

📝 Content preview (first 1000 chars):
LangChain overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain is an o

# Loading Text and Markdown Files

In [18]:
from langchain_community.document_loaders import TextLoader

# Load notes.txt file
txt_path = '/content/drive/MyDrive/Datasets/sample_data/notes.txt'

if Path(txt_path).exists():
  print(f"Loading text file:{txt_path}\n")

  # Create Loader
  loader = TextLoader(txt_path, encoding='utf-8')

  # Load the file
  documents = loader.load()

  print(f"✅ Loaded {len(documents)} document\n")

  doc=documents[0]
  print(f"📄 Content length:{len(doc.page_content)} characters")
  print(f"\n📝 First 300 characters:\n{doc.page_content[:300]}...")
  print(f"\n🔍 Metadata:{doc.metadata}")

else:
  print(f"❌ Text file not found:{txt_path}")

Loading text file:/content/drive/MyDrive/Datasets/sample_data/notes.txt

✅ Loaded 1 document

📄 Content length:8567 characters

📝 First 300 characters:
LANGCHAIN STUDY NOTES - RAG IMPLEMENTATION

Date: January 15, 2025
Topic: Retrieval-Augmented Generation with LangChain 1.0+


CORE CONCEPTS
-------------

1. Document Object Structure
   - page_content: The actual text content
   - metadata: Dictionary wit...

🔍 Metadata:{'source': '/content/drive/MyDrive/Datasets/sample_data/notes.txt'}


## Markdown Files

In [21]:
readme_path = "/content/drive/MyDrive/Datasets/sample_data/README.md"

if Path(readme_path).exists():
  try:
    from langchain_community.document_loaders import UnstructuredMarkdownLoader

    print(f"Loading Markdown:{readme_path}\n")

    loader = UnstructuredMarkdownLoader(readme_path)
    docs = loader.load()

    print(f"✅ Loaded {len(docs)} document(s)")
    print(f"\nFirst 200 chars:\n{docs[0].page_content[:200]}...")

  except ImportError:
    print(f"⚠️ UnstructuredMarkdownLoader requires additional dependencies")
    print(" Install with: pip install unstructured")
    print("\n For now, using TextLoader:")

    loader = TextLoader(readme_path)
    docs = loader.load()
    print(f"  ✅ Loaded with TextLoader:{len(docs[0].page_content)} chars")

else:
  print("ℹ️ No README.md found in current directory")

Loading Markdown:/content/drive/MyDrive/Datasets/sample_data/README.md

✅ Loaded 1 document(s)

First 200 chars:
рҹӨ– Flan-T5 Text Summarization вҖ” MLOps Project

Fine-tune google/flan-t5-base on the SAMSum dialogue dataset, deploy a FastAPI inference API to AWS App Runner, and log every inference to W&B Weave....


# Batch Loading with DirectoryLoader

In [24]:
from langchain_community.document_loaders import DirectoryLoader

data_dir = "/content/drive/MyDrive/Datasets/sample_data"

if Path(data_dir).exists():
  print(f"📂 Loading all text files from:{data_dir}\n")
  loader = DirectoryLoader(
      data_dir,
      glob="*.txt",
      loader_cls=TextLoader,
      show_progress=True
  )

  documents = loader.load()

  print(f"\n✅ Loaded {len(documents)} text file(s)\n")

  for doc in documents:
    print(f"  -{doc.metadata['source']} ({len(doc.page_content)} chars)")
else:
    print(f"❌ Directory not found:{data_dir}")

📂 Loading all text files from:/content/drive/MyDrive/Datasets/sample_data



100%|██████████| 1/1 [00:00<00:00, 281.29it/s]


✅ Loaded 1 text file(s)

  -/content/drive/MyDrive/Datasets/sample_data/notes.txt (8567 chars)


In [25]:
# Advanced: Load all files from a directory (mixed types)
# This function handles different file types intelligently

def load_all_documents(directory: str) -> list:
    """
    Load documents from multiple file formats in a directory.

    Supports: PDF, TXT, CSV, JSON, HTML
    """
    all_docs = []
    directory_path = Path(directory)

    if not directory_path.exists():
        print(f"❌ Directory not found: {directory}")
        return []

    print(f"📂 Loading from: {directory}\n")

    # Load PDFs
    pdf_files = list(directory_path.glob("*.pdf"))
    for pdf in pdf_files:
        loader = PyPDFLoader(str(pdf))
        docs = loader.load()
        all_docs.extend(docs)
        print(f"  ✅ PDF: {pdf.name} ({len(docs)} pages)")

    # Load TXT files
    txt_files = list(directory_path.glob("*.txt"))
    for txt in txt_files:
        loader = TextLoader(str(txt))
        docs = loader.load()
        all_docs.extend(docs)
        print(f"  ✅ TXT: {txt.name}")

    # Load CSV files
    csv_files = list(directory_path.glob("*.csv"))
    for csv in csv_files:
        loader = CSVLoader(str(csv))
        docs = loader.load()
        all_docs.extend(docs)
        print(f"  ✅ CSV: {csv.name} ({len(docs)} rows)")

    # Load JSON files
    json_files = list(directory_path.glob("*.json"))
    for json_file in json_files:
        try:
            loader = JSONLoader(
                str(json_file),
                jq_schema=".",
                text_content=False
            )
            docs = loader.load()
            all_docs.extend(docs)
            print(f"  ✅ JSON: {json_file.name}")
        except Exception as e:
            print(f"  ⚠️ JSON: {json_file.name} (error: {str(e)[:50]}...)")

    print(f"\n📊 Total: {len(all_docs)} documents loaded")
    return all_docs

# Test the function
if Path("sample_data").exists():
    all_documents = load_all_documents("sample_data")

    # Show summary
    print(f"\n📈 Summary:")
    sources = [doc.metadata['source'] for doc in all_documents]
    print(f"   Files loaded: {len(set(sources))}")
    print(f"   Total documents: {len(all_documents)}")

📂 Loading from: sample_data

  ✅ CSV: mnist_train_small.csv (19999 rows)
  ✅ CSV: california_housing_test.csv (3000 rows)
  ✅ CSV: california_housing_train.csv (17000 rows)
  ✅ CSV: mnist_test.csv (9999 rows)
  ✅ JSON: anscombe.json

📊 Total: 49999 documents loaded

📈 Summary:
   Files loaded: 5
   Total documents: 49999
